# 07 — Catchment assignment (Stage 4b)

For each LSOA21 population-weighted centroid, find the nearest school per phase on the walking network, capped at the per-phase walking distance (DEC-016: 2 km primary, 5 km secondary, 5 km special).

Outputs:
- `data/processed/assignments_nearest_<date>.parquet` — primary, hard-nearest assignment
- `data/processed/assignments_knn_<date>.parquet` — k=3 IDW sensitivity (DEC-011)

Both are long-format: one row per `(lsoa21cd, urn, phase, distance_m, weight, pop_school_age)`.

## 0. Pre-flight

In [ ]:
import time
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, catchment, config, network as nw

config.ensure_dirs()
audit.verify_manifest()

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
TODAY

## 1. Load the walking network and rebuild the pandana graph

(Reloading from parquet is fast — ~1–3 s for the parsed nodes / edges.)

In [ ]:
net_dirs = sorted(p for p in config.DATA_PROCESSED.glob("walk_network_*") if p.is_dir())
if not net_dirs:
    raise RuntimeError("No walk_network_* directory in data/processed/. Run notebook 06 first.")
net_dir = net_dirs[-1]
print("Using network at:", net_dir.name)

data = nw.load_walking_network(net_dir)
print(f"  Nodes: {len(data.nodes):,}, edges: {len(data.edges):,}")

t0 = time.time()
net = nw.build_pandana_network(data)
# pandana.Network constructor builds the contraction hierarchy already; we
# do not call .precompute() because set_pois() does the per-category work
# and the explicit precompute on a 1.5M-node graph was OOM-prone on macOS.
print(f"Network ready in {time.time()-t0:.1f}s")

## 2. Load schools and LSOA PWCs

In [ ]:
schools_files = sorted(
    p for p in config.DATA_PROCESSED.glob("schools_ne_*.gpkg") if "sensitivity" not in p.name
)
if not schools_files:
    raise RuntimeError("No schools_ne_*.gpkg in data/processed/. Run notebook 01 first.")
schools = gpd.read_file(schools_files[-1])
print(f"Loaded {schools_files[-1].name}: {len(schools)} schools")

lsoa_files = sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))
lsoa_attrs = gpd.read_file(lsoa_files[-1])

pwc_files = sorted(config.DATA_PROCESSED.glob("lsoa_pwc_ne_*.gpkg"))
pwc = gpd.read_file(pwc_files[-1])
# Carry the school-age population on the PWCs so assignment outputs include it.
pwc = pwc.merge(
    lsoa_attrs[["lsoa21cd", "lad_code", "pop_school_age", "imd_quintile"]],
    on="lsoa21cd",
    how="left",
)
print(f"Loaded PWCs: {len(pwc)} (school-age pop range {pwc['pop_school_age'].min()}–{pwc['pop_school_age'].max()})")

## 3. Hard nearest-school assignment (primary, DEC-009)

In [ ]:
t0 = time.time()
result = catchment.assign_nearest_school(net, pwc, schools)
print(f"Assigned in {time.time()-t0:.1f}s")
for k, v in result.audit.items():
    print(f"  {k:>26s}: {v}")
result.assignments.head()

In [ ]:
# Distance distributions per phase
result.assignments.groupby("phase_cat")["distance_m"].describe().round(0)

In [ ]:
# % of LSOAs with a within-cap match per phase
n_lsoa = len(pwc)
for phase_cat in ("primary", "secondary", "special"):
    sub = result.assignments[result.assignments["phase_cat"] == phase_cat]
    print(f"{phase_cat:>10s}: {sub['lsoa21cd'].nunique():4d} / {n_lsoa} LSOAs assigned ({sub['lsoa21cd'].nunique()*100/n_lsoa:.1f}%)")

## 4. k=3 IDW sensitivity (DEC-011, DEC-015)

In [ ]:
t0 = time.time()
knn = catchment.assign_knn_idw(net, pwc, schools, k=config.KNN_K_SENSITIVITY)
print(f"k-NN IDW assignment in {time.time()-t0:.1f}s")
for k, v in knn.audit.items():
    print(f"  {k:>22s}: {v}")
print()
print("Rows per LSOA × phase (should average roughly k where supply allows):")
knn.assignments.groupby(["lsoa21cd", "phase_cat"]).size().describe().round(2).to_frame("rows_per_pair")

## 5. Persist outputs

In [ ]:
out_nearest = config.DATA_PROCESSED / f"assignments_nearest_{TODAY}.parquet"
out_knn = config.DATA_PROCESSED / f"assignments_knn_{TODAY}.parquet"
result.assignments.to_parquet(out_nearest, index=False)
knn.assignments.to_parquet(out_knn, index=False)

for path, inputs, notes in (
    (out_nearest, (schools_files[-1], pwc_files[-1], lsoa_files[-1]), "Hard-nearest school assignment, capped per phase (DEC-009 + DEC-016)."),
    (out_knn, (schools_files[-1], pwc_files[-1], lsoa_files[-1]), f"k=3 IDW sensitivity assignment (DEC-011)."),
):
    audit.write_provenance_sidecar(
        audit.Provenance(
            output_path=path,
            inputs=inputs,
            notes=notes,
            random_seed=config.RANDOM_SEED,
        )
    )
    print("wrote:", path.name)

## 6. Quick-look — one neighbourhood

For one LSOA in central Newcastle, list its assigned schools across all phases. Sanity check: distances should be modest, school names should be plausibly local.

In [ ]:
newcastle = pwc.loc[pwc["lad_code"] == "E08000021"].sort_values("pop_school_age", ascending=False).head(1)
if len(newcastle):
    sample_lsoa = newcastle.iloc[0]["lsoa21cd"]
    print(f"LSOA: {sample_lsoa} (Newcastle, top-pop)")
    rows = result.assignments[result.assignments["lsoa21cd"] == sample_lsoa]
    rows = rows.merge(schools[["urn", "establishment_name"]], on="urn", how="left")
    print(rows[["phase_cat", "phase", "establishment_name", "distance_m"]].to_string(index=False))

## Done

Outputs:
- `data/processed/assignments_nearest_<date>.parquet` (primary)
- `data/processed/assignments_knn_<date>.parquet` (sensitivity)

Stage 5 (`08_routes_and_buffers.ipynb`) computes shortest-path geometries for each (lsoa, urn) pair in the primary assignment and buffers them at 50 m / 100 m.